In [ ]:
# ==============================================================================
# Interactive Demo for the Fine-Tuned Legal Summarizer with Translation
# ==============================================================================
# This notebook loads the pre-trained model from Google Drive and provides
# an interactive interface to generate summaries and optionally translate them
# into different languages using the Gemini API.

# --- 1. Install necessary libraries (quietly) ---
!pip install -q transformers[torch] ipywidgets

# --- 2. Import required libraries ---
import torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import ipywidgets as widgets
from IPython.display import display
import json
import urllib.request
import time # Import the time module for handling retries

# --- 3. Connect to Google Drive ---
# This is where your saved model is stored.
print("Connecting to Google Drive...")
try:
    drive.mount('/content/drive', force_remount=True)
    print("Connected.")
except Exception as e:
    print(f"Error connecting to Drive: {e}")


# --- 4. Load the Fine-Tuned Model and Tokenizer ---
# Make sure this path is the EXACT same as the one you used for saving.
MODEL_PATH = "/content/drive/MyDrive/fine_tuned_legal_summarizer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREFIX = "summarize: "

print(f"Loading summarization model from: {MODEL_PATH}")
print(f"Using device: {DEVICE}")

# Load the tokenizer and model from your Google Drive
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
    print("Summarization model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please make sure the path is correct and the model files are in your Google Drive.")

# --- 5. Create the Summarization and Translation Functions ---
def summarize_text(document):
    """Generates a summary for a given text document."""
    if not document.strip():
        return "Please enter some text to summarize."

    inputs = tokenizer(
        PREFIX + document, return_tensors="pt", max_length=1024, truncation=True
    ).to(DEVICE)
    summary_ids = model.generate(
        inputs["input_ids"], num_beams=4, max_length=256, early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def translate_text_with_gemini(text_to_translate, target_language, api_key):
    """Translates text using the Gemini API."""
    if not api_key:
        return "Error: Gemini API Key is missing."

    API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-preview-09-2025:generateContent?key={api_key}"

    # Create a clear prompt for the model
    prompt = f"Translate the following English text to {target_language}. Respond with only the translated text, nothing else:\n\n---\n\n{text_to_translate}"

    payload = {
        "contents": [{"parts": [{"text": prompt}]}]
    }

    try:
        req = urllib.request.Request(API_URL, data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode("utf-8"))
            # Navigate the JSON response to get the translated text
            return result['candidates'][0]['content']['parts'][0]['text'].strip()
    except Exception as e:
        return f"Error during translation: {e}"


# --- 6. Create an Interactive User Interface ---
print("\n--- Legal Document Summarizer & Translator ---")
print("You can get a Gemini API key from Google AI Studio.")
print("Enter a legal document, choose a language (optional), and click 'Generate Summary'.")

# API Key input
api_key_input = widgets.Password(
    description='Gemini API Key:',
    placeholder='Enter your key here',
    layout={'width': '50%'}
)

# Text area for user input
input_text = widgets.Textarea(
    placeholder='Paste a long legal document here...',
    layout={'height': '300px', 'width': '95%'}
)

# Dropdown for language selection
language_dropdown = widgets.Dropdown(
    options=['None', 'Spanish', 'French', 'German', 'Hindi', 'Japanese', 'Mandarin Chinese', 'Russian', 'Telugu'],
    value='None',
    description='Translate to:',
)

# Button to trigger summarization
summarize_button = widgets.Button(
    description='Generate Summary',
    button_style='success',
    tooltip='Click to summarize the text above'
)

# Area to display the output
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        output_area.clear_output()
        print("Summarizing, please wait...")

        # 1. Generate the summary
        summary = summarize_text(input_text.value)

        output_area.clear_output()
        print("--- Generated Summary (English) ---")
        print(summary)

        # 2. Check if translation is needed
        target_language = language_dropdown.value
        if target_language != 'None':
            print(f"\n--- Translating to {target_language}, please wait... ---")
            api_key = api_key_input.value
            translated_summary = translate_text_with_gemini(summary, target_language, api_key)
            print(translated_summary)

summarize_button.on_click(on_button_clicked)

# Display the UI elements
display(api_key_input, input_text, language_dropdown, summarize_button, output_area)


